# 00 - Environment and Raw Data Audit

This is the first learning notebook for the **Smart City Traffic Intelligence** capstone.

## Learning objectives

By the end of this notebook, you will be able to:

1. Check which Python environment is running the notebook.
2. Find the project and CSV file without hard-coded absolute paths.
3. Load and inspect a CSV using Python's standard library.
4. Validate the required schema before processing data.
5. Detect blanks, duplicates, invalid dates and suspicious numerical values.
6. Confirm the audit with automated assertions.

> Run the notebook from top to bottom using **Run All**. Read the explanation above each code cell before running it. This notebook is a learning draft and is not yet a final graded pipeline.

## 1. Check the Python environment

A reproducible project records the Python executable and version used to run the analysis. If two people use different environments, they may get different results.

In [1]:
import csv
import platform
import sys
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from pprint import pprint

print(f"Python version: {platform.python_version()}")
print(f"Python executable: {sys.executable}")
print(f"Operating system: {platform.platform()}")

Python version: 3.11.9
Python executable: /Users/husa/software/AI/nus course/Capstone Project/.venv/bin/python
Operating system: macOS-27.0.1-arm64-arm-64bit


## 2. Locate the project and raw CSV

The notebook might start from the repository root or from `notebooks/learning`. The function below searches the current directory and its parents for the supplied CSV. This is more portable than using a path tied to one computer.

In [2]:
CSV_FILENAME = "Metro_Interstate_Traffic_Volume.csv"

def find_project_root(start: Path, filename: str) -> Path:
    candidates = (start.resolve(), *start.resolve().parents)
    for candidate in candidates:
        if (candidate / filename).is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename!r} from {start.resolve()} or its parent directories."
    )

PROJECT_ROOT = find_project_root(Path.cwd(), CSV_FILENAME)
CSV_PATH = PROJECT_ROOT / CSV_FILENAME

print(f"Project root: {PROJECT_ROOT}")
print(f"CSV path: {CSV_PATH}")
print(f"CSV size: {CSV_PATH.stat().st_size:,} bytes")

Project root: /Users/husa/software/AI/nus course/Capstone Project
CSV path: /Users/husa/software/AI/nus course/Capstone Project/Metro_Interstate_Traffic_Volume.csv
CSV size: 3,237,208 bytes


## 3. Load the CSV

`csv.DictReader` returns each record as a dictionary whose keys come from the header row. The `utf-8-sig` encoding also handles a possible byte-order mark safely.

In [3]:
with CSV_PATH.open(mode="r", encoding="utf-8-sig", newline="") as csv_file:
    reader = csv.DictReader(csv_file)
    rows = list(reader)
    columns = reader.fieldnames or []

print(f"Rows loaded: {len(rows):,}")
print(f"Columns loaded: {len(columns)}")
print(f"Columns: {columns}")
print("\nFirst three records:")
pprint(rows[:3], sort_dicts=False)

Rows loaded: 48,204
Columns loaded: 9
Columns: ['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'date_time', 'traffic_volume']

First three records:
[{'holiday': 'None',
  'temp': '288.28',
  'rain_1h': '0.0',
  'snow_1h': '0.0',
  'clouds_all': '40',
  'weather_main': 'Clouds',
  'weather_description': 'scattered clouds',
  'date_time': '2012-10-02 09:00:00',
  'traffic_volume': '5545'},
 {'holiday': 'None',
  'temp': '289.36',
  'rain_1h': '0.0',
  'snow_1h': '0.0',
  'clouds_all': '75',
  'weather_main': 'Clouds',
  'weather_description': 'broken clouds',
  'date_time': '2012-10-02 10:00:00',
  'traffic_volume': '4516'},
 {'holiday': 'None',
  'temp': '289.58',
  'rain_1h': '0.0',
  'snow_1h': '0.0',
  'clouds_all': '90',
  'weather_main': 'Clouds',
  'weather_description': 'overcast clouds',
  'date_time': '2012-10-02 11:00:00',
  'traffic_volume': '4767'}]


## 4. Validate the schema before processing

The capstone instructions require schema validation before cleaning. We compare the actual header with the nine expected columns. Set comparison identifies missing or unexpected fields, while list comparison also detects an unexpected order.

In [4]:
EXPECTED_COLUMNS = [
    "holiday",
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "weather_main",
    "weather_description",
    "date_time",
    "traffic_volume",
]

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(columns))
unexpected_columns = sorted(set(columns) - set(EXPECTED_COLUMNS))
column_order_matches = columns == EXPECTED_COLUMNS

print(f"Missing columns: {missing_columns}")
print(f"Unexpected columns: {unexpected_columns}")
print(f"Expected column order: {column_order_matches}")

if missing_columns or unexpected_columns:
    raise ValueError("The CSV schema does not match the expected capstone schema.")

Missing columns: []
Unexpected columns: []
Expected column order: True


## 5. Check blank fields

A blank field is not the same as the text `None`. In this dataset, `None` in the holiday column means the hour was not labelled as a holiday and should initially remain a valid category.

In [5]:
blank_counts = {
    column: sum(not row[column].strip() for row in rows)
    for column in columns
}

print("Blank fields by column:")
pprint(blank_counts, sort_dicts=False)
print(f"Total blank fields: {sum(blank_counts.values())}")

Blank fields by column:
{'holiday': 0,
 'temp': 0,
 'rain_1h': 0,
 'snow_1h': 0,
 'clouds_all': 0,
 'weather_main': 0,
 'weather_description': 0,
 'date_time': 0,
 'traffic_volume': 0}
Total blank fields: 0


## 6. Check exact duplicates and repeated timestamps

These are two different data-quality questions:

- An **exact duplicate** repeats every field and can normally be removed.
- A **repeated timestamp** may contain another weather observation for the same hour. It must be investigated before removal or aggregation.

In [6]:
row_signatures = [tuple(row[column] for column in columns) for row in rows]
exact_duplicate_count = len(row_signatures) - len(set(row_signatures))

timestamp_counts = Counter(row["date_time"] for row in rows)
unique_timestamp_count = len(timestamp_counts)
repeated_timestamp_rows = sum(
    count for count in timestamp_counts.values() if count > 1
)
additional_timestamp_records = sum(
    count - 1 for count in timestamp_counts.values() if count > 1
)
maximum_records_in_one_hour = max(timestamp_counts.values())

print(f"Exact duplicate rows: {exact_duplicate_count:,}")
print(f"Unique timestamps: {unique_timestamp_count:,}")
print(f"Rows involved in repeated timestamps: {repeated_timestamp_rows:,}")
print(f"Additional timestamp records: {additional_timestamp_records:,}")
print(f"Maximum records for one timestamp: {maximum_records_in_one_hour}")

Exact duplicate rows: 17
Unique timestamps: 40,575
Rows involved in repeated timestamps: 13,074
Additional timestamp records: 7,629
Maximum records for one timestamp: 6


## 7. Parse and validate the date/time field

Parsing converts date text into real `datetime` objects. Invalid values are collected with their CSV line numbers so that a future pipeline can log useful error details.

In [7]:
parsed_dates = []
invalid_dates = []

for csv_line_number, row in enumerate(rows, start=2):
    try:
        parsed_dates.append(
            datetime.strptime(row["date_time"], "%Y-%m-%d %H:%M:%S")
        )
    except ValueError as exc:
        invalid_dates.append(
            {
                "line": csv_line_number,
                "value": row["date_time"],
                "error": str(exc),
            }
        )

print(f"Invalid date/time values: {len(invalid_dates)}")
if invalid_dates:
    pprint(invalid_dates[:5], sort_dicts=False)
else:
    print(f"Earliest timestamp: {min(parsed_dates)}")
    print(f"Latest timestamp: {max(parsed_dates)}")

Invalid date/time values: 0
Earliest timestamp: 2012-10-02 09:00:00
Latest timestamp: 2018-09-30 23:00:00


## 8. Convert numerical fields and detect suspicious values

The raw CSV stores values as text. Before numerical analysis, the pipeline must convert them and capture conversion errors. We then flag known concerns from the assignment: `0 K` temperatures and extremely high rainfall. Zero traffic is also reported for investigation, but it is not automatically declared invalid.

In [8]:
NUMERIC_COLUMNS = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "traffic_volume",
]

numeric_values = {column: [] for column in NUMERIC_COLUMNS}
numeric_conversion_errors = []

for csv_line_number, row in enumerate(rows, start=2):
    for column in NUMERIC_COLUMNS:
        try:
            numeric_values[column].append(float(row[column]))
        except ValueError as exc:
            numeric_conversion_errors.append(
                {
                    "line": csv_line_number,
                    "column": column,
                    "value": row[column],
                    "error": str(exc),
                }
            )

zero_kelvin_count = sum(value == 0 for value in numeric_values["temp"])
extreme_rain_count = sum(value > 9000 for value in numeric_values["rain_1h"])
zero_traffic_count = sum(value == 0 for value in numeric_values["traffic_volume"])

print(f"Numeric conversion errors: {len(numeric_conversion_errors)}")
print(f"Temperature records equal to 0 K: {zero_kelvin_count}")
print(f"Rainfall records above 9,000 mm: {extreme_rain_count}")
print(f"Traffic-volume records equal to zero: {zero_traffic_count}")

print("\nObserved numerical ranges:")
for column, values in numeric_values.items():
    print(f"{column:>14}: {min(values):>8g} to {max(values):>8g}")

Numeric conversion errors: 0
Temperature records equal to 0 K: 10
Rainfall records above 9,000 mm: 1
Traffic-volume records equal to zero: 2

Observed numerical ranges:
          temp:        0 to   310.07
       rain_1h:        0 to   9831.3
       snow_1h:        0 to     0.51
    clouds_all:        0 to      100
traffic_volume:        0 to     7280


## 9. Inspect categorical consistency

Categories that differ only by capitalisation can incorrectly appear as separate groups. This cell lists all unique high-level weather categories and identifies case-insensitive duplicates in the detailed descriptions.

In [9]:
weather_main_counts = Counter(row["weather_main"] for row in rows)
weather_description_variants = defaultdict(set)

for row in rows:
    description = row["weather_description"].strip()
    weather_description_variants[description.casefold()].add(description)

case_inconsistent_descriptions = {
    normalised: sorted(variants)
    for normalised, variants in weather_description_variants.items()
    if len(variants) > 1
}

print("weather_main counts:")
pprint(weather_main_counts.most_common(), sort_dicts=False)
print("\nCase-inconsistent weather descriptions:")
pprint(case_inconsistent_descriptions, sort_dicts=False)

weather_main counts:
[('Clouds', 15164),
 ('Clear', 13391),
 ('Mist', 5950),
 ('Rain', 5672),
 ('Snow', 2876),
 ('Drizzle', 1821),
 ('Haze', 1360),
 ('Thunderstorm', 1034),
 ('Fog', 912),
 ('Smoke', 20),
 ('Squall', 4)]

Case-inconsistent weather descriptions:
{'sky is clear': ['Sky is Clear', 'sky is clear']}


## 10. Audit coverage by year

The assignment requests annual totals, but several years are incomplete. Counting both records and unique hours helps prevent missing coverage or repeated weather observations from being mistaken for genuine traffic growth or decline.

In [10]:
year_records = Counter(date.year for date in parsed_dates)
year_unique_timestamps = defaultdict(set)
year_dates = defaultdict(list)

for row, parsed_date in zip(rows, parsed_dates):
    year_unique_timestamps[parsed_date.year].add(row["date_time"])
    year_dates[parsed_date.year].append(parsed_date)

print(f"{'Year':<6}{'Records':>10}{'Unique hours':>15}  Coverage")
for year in sorted(year_records):
    first = min(year_dates[year]).strftime("%Y-%m-%d %H:%M")
    last = max(year_dates[year]).strftime("%Y-%m-%d %H:%M")
    print(
        f"{year:<6}{year_records[year]:>10,}"
        f"{len(year_unique_timestamps[year]):>15,}  {first} to {last}"
    )

Year     Records   Unique hours  Coverage
2012       2,559          2,103  2012-10-02 09:00 to 2012-12-31 23:00
2013       8,573          7,294  2013-01-01 00:00 to 2013-12-31 23:00
2014       4,839          4,501  2014-01-01 00:00 to 2014-08-08 01:00
2015       4,373          3,593  2015-06-11 20:00 to 2015-12-31 23:00
2016       9,306          7,838  2016-01-01 00:00 to 2016-12-31 23:00
2017      10,605          8,713  2017-01-01 00:00 to 2017-12-31 23:00
2018       7,949          6,533  2018-01-01 00:00 to 2018-09-30 23:00


## 11. Produce a compact audit summary

A compact dictionary is useful for logging, tests and later comparison with the Pandas pipeline.

In [11]:
audit_summary = {
    "row_count": len(rows),
    "column_count": len(columns),
    "blank_field_count": sum(blank_counts.values()),
    "exact_duplicate_count": exact_duplicate_count,
    "unique_timestamp_count": unique_timestamp_count,
    "additional_timestamp_records": additional_timestamp_records,
    "invalid_date_count": len(invalid_dates),
    "numeric_conversion_error_count": len(numeric_conversion_errors),
    "zero_kelvin_count": zero_kelvin_count,
    "extreme_rain_count": extreme_rain_count,
    "zero_traffic_count": zero_traffic_count,
    "earliest_timestamp": min(parsed_dates).isoformat(sep=" "),
    "latest_timestamp": max(parsed_dates).isoformat(sep=" "),
}

pprint(audit_summary, sort_dicts=False)

{'row_count': 48204,
 'column_count': 9,
 'blank_field_count': 0,
 'exact_duplicate_count': 17,
 'unique_timestamp_count': 40575,
 'additional_timestamp_records': 7629,
 'invalid_date_count': 0,
 'numeric_conversion_error_count': 0,
 'zero_kelvin_count': 10,
 'extreme_rain_count': 1,
 'zero_traffic_count': 2,
 'earliest_timestamp': '2012-10-02 09:00:00',
 'latest_timestamp': '2018-09-30 23:00:00'}


## 12. Run automated checks

Assertions turn expected findings into executable checks. A failed assertion means the supplied file differs from the file reviewed during project planning or an earlier calculation is incorrect.

In [12]:
EXPECTED_AUDIT_VALUES = {
    "row_count": 48_204,
    "column_count": 9,
    "blank_field_count": 0,
    "exact_duplicate_count": 17,
    "unique_timestamp_count": 40_575,
    "additional_timestamp_records": 7_629,
    "invalid_date_count": 0,
    "numeric_conversion_error_count": 0,
    "zero_kelvin_count": 10,
    "extreme_rain_count": 1,
    "zero_traffic_count": 2,
}

for key, expected_value in EXPECTED_AUDIT_VALUES.items():
    actual_value = audit_summary[key]
    assert actual_value == expected_value, (
        f"{key} was {actual_value!r}; expected {expected_value!r}"
    )

assert columns == EXPECTED_COLUMNS, "The column names or order changed."
assert audit_summary["earliest_timestamp"] == "2012-10-02 09:00:00"
assert audit_summary["latest_timestamp"] == "2018-09-30 23:00:00"

print("PASS - All raw-data audit checks matched the expected dataset.")

PASS - All raw-data audit checks matched the expected dataset.


## 13. Reflection before production code

Before moving this work into the final pipeline, consider these questions:

1. Why should exact duplicate rows and repeated timestamps be handled differently?
2. Why might annual totals be misleading when yearly coverage differs?
3. Why should a zero traffic value be investigated rather than automatically imputed?
4. Why is `None` in the holiday field not treated as a blank value?
5. Which cleaning decisions must be logged as warnings in the final pipeline?

### Stop point

If the last code cell reports `PASS`, save the notebook and review its results. We will discuss the findings before creating the final production pipeline or making a Git commit.